# Práctica 2 – Aprendizaje de Máquinas (Introducción a la IA – 3010476)
**Dominio:** Calidad del aire en India (Kaggle: `rohanrao/air-quality-data-in-india`, archivo `city_day.csv`)
**Tarea:** clasificar `AQI_Bucket` (6 clases) y luego agrupar sin etiqueta.

Estructura del notebook (módulos):
0. Configuración y utilidades · 1. Datos y preprocesamiento · 2. Aprendizaje supervisado · 3. Aprendizaje no supervisado

> **Para cambiar la cédula / semilla / tamaño basta con editar la celda de configuración (sección 0).**

## 0. Configuración, instalación y utilidades

In [ ]:
# Instalación de librerías (Colab ya trae casi todo; imbalanced-learn y kagglehub suelen faltar)
%pip install -q imbalanced-learn kagglehub pandas matplotlib seaborn tensorflow

In [ ]:
import os, random, warnings, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score,
                             precision_recall_curve, auc, silhouette_score)
from imblearn.over_sampling import SMOTE, RandomOverSampler

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

### 0.1 Parámetros del equipo (EDITAR AQUÍ)
* `CEDULA`: cédula del integrante elegido (obligatorio declararla en la entrega).
* Tamaño del subconjunto = 5000 + (último dígito ≠ 0) × 150.
* Semilla = últimos dos dígitos ≠ 0 de la cédula (no puede ser 42).

In [ ]:
CEDULA = "1025645721"
GRUPO, EQUIPO = "01", "10"

# Interruptores de ejecución
SCALER = "standard"        # "standard" | "minmax"  (justificación en la sección 1)
BALANCE_METHOD = "smote"   # "smote" | "ros" (sobremuestreo aleatorio)
VIZ_ALL_CASES = True       # visualizaciones 2.4 para los 8 casos (se guardan en salidas/; solo el caso 8 se muestra en pantalla)
DATA_PATH = "city_day.csv"  # ruta local del CSV (si no existe se intenta kagglehub y luego subida manual en Colab)
OUT_DIR = "salidas"        # tablas y figuras
os.makedirs(OUT_DIR, exist_ok=True)


def derivar_parametros(cedula: str):
    """Devuelve (n_registros, seed) a partir de la cédula según el enunciado."""
    nz = [c for c in str(cedula) if c.isdigit() and c != "0"]
    if len(nz) < 2:
        raise ValueError("La cédula necesita al menos dos dígitos distintos de cero.")
    n = 5000 + int(nz[-1]) * 150
    seed = int("".join(nz[-2:]))
    if seed == 42:
        raise ValueError("El enunciado prohíbe usar la semilla 42: elijan otra cédula.")
    return n, seed


N_REGISTROS, SEED = derivar_parametros(CEDULA)
print(f"Cédula: {CEDULA} -> N_REGISTROS = {N_REGISTROS}, SEED (random_state) = {SEED}")


def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seeds()

### 0.2 Constantes del dominio

In [ ]:
TARGET = "AQI_Bucket"
# Orden por severidad (de menor a mayor contaminación)
CLASES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASES)}
K_CLASES = len(CLASES)
NUM_COLS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]
NOMINAL_COLS = ["City", "Season"]   # -> One-Hot
ORDINAL_COLS = ["Year"]             # -> codificación ordinal (orden temporal). Justificar en el informe.
# 'AQI' se EXCLUYE: AQI_Bucket se deriva directamente de AQI (fuga de información).

# Los 8 casos de la figura 1 / figura 2
CASOS = {i + 1: dict(scale=(i >= 4), outliers=(i % 4 >= 2), balance=(i % 2 == 1)) for i in range(8)}
for k, v in CASOS.items():
    print(k, "ED(SI)" if v["scale"] else "ED(NO)", "| outliers:", "SI" if v["outliers"] else "NO",
          "| balanceo:", "SI" if v["balance"] else "NO")


def savefig(name, show=True):
    """Guarda la figura en OUT_DIR y la muestra (o la cierra si show=False)."""
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, name), dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close()

## 1. Datos y preprocesamiento (módulo común)
### 1.1 Carga del dataset

In [ ]:
def load_data():
    """Busca city_day.csv local; si no, intenta kagglehub; si no, pide subirlo (Colab)."""
    for p in [DATA_PATH, "/content/city_day.csv", "data/city_day.csv"]:
        if os.path.exists(p):
            return pd.read_csv(p)
    try:
        import kagglehub
        path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")
        return pd.read_csv(os.path.join(path, "city_day.csv"))  # dataset público; si pide login: kagglehub.login()
    except Exception as e:
        print("kagglehub falló:", e)
    from google.colab import files  # solo en Colab
    up = files.upload()
    return pd.read_csv(next(iter(up)))


raw = load_data()
print("Dataset original:", raw.shape)
raw.head()

### 1.2 Subconjunto único (N según cédula) e ingeniería de variables
Se descartan registros sin etiqueta y se toma una muestra aleatoria de `N_REGISTROS` filas con la semilla definida.
De `Date` se derivan `Season` (nominal) y `Year` (ordinal).

In [ ]:
def season_of(m):
    return {12: "Winter", 1: "Winter", 2: "Winter", 3: "Summer", 4: "Summer", 5: "Summer",
            6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon", 10: "PostMonsoon", 11: "PostMonsoon"}[m]


def feature_engineering(df):
    df = df.copy()
    d = pd.to_datetime(df["Date"])
    df["Year"] = d.dt.year
    df["Season"] = d.dt.month.map(season_of)
    return df[["City", "Season", "Year"] + NUM_COLS + [TARGET]]


labeled = raw.dropna(subset=[TARGET])
assert len(labeled) >= N_REGISTROS, "No hay suficientes registros etiquetados"
sub = labeled.sample(n=N_REGISTROS, random_state=SEED).reset_index(drop=True)
sub = feature_engineering(sub)
YEARS_ORDER = sorted(sub["Year"].unique())   # orden fijo de la variable ordinal
print(f"Subconjunto: {sub.shape} | clases: {sub[TARGET].nunique()} (mínimo exigido: 4, registros >= 5000: {len(sub) >= 5000})")

### 1.3 Análisis exploratorio (dimensiones, tipos, clases, estadísticas, nulos, correlación)

In [ ]:
def eda(df):
    print("Dimensiones:", df.shape)
    print("\nTipos de variables:\n", df.dtypes.to_string())
    print("\nDistribución de clases:\n", df[TARGET].value_counts().reindex(CLASES).to_frame("n").assign(
        pct=lambda t: (100 * t.n / t.n.sum()).round(2)).to_string())
    print("\nEstadísticas descriptivas:\n", df[NUM_COLS].describe().T.round(2).to_string())
    nulls = df.isna().sum().to_frame("nulos").assign(pct=lambda t: (100 * t.nulos / len(df)).round(2))
    print("\nNulos por variable:\n", nulls.to_string())
    plt.figure(figsize=(9, 7))
    sns.heatmap(df[NUM_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
    plt.title("Matriz de correlación (variables numéricas)")
    savefig("eda_correlacion.png")
    df[TARGET].value_counts().reindex(CLASES).plot(kind="bar", color="steelblue", figsize=(7, 3.5), title="Distribución de AQI_Bucket")
    savefig("eda_clases.png")


eda(sub)

### 1.4 Partición estratificada 70/15/15 e imputación
La partición se hace una sola vez (mismo test para los 8 casos). La imputación (mediana para numéricas, moda para categóricas)
se ajusta **solo con train** y se aplica a validación y prueba.

In [ ]:
def split_data(df, seed=SEED):
    tr, tmp = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp[TARGET], random_state=seed)
    return tr.copy(), va.copy(), te.copy()


def fit_imputer(train):
    return {**{c: train[c].median() for c in NUM_COLS}, **{c: train[c].mode()[0] for c in NOMINAL_COLS + ORDINAL_COLS}}


def apply_imputer(df, imp):
    return df.fillna(imp)


tr_raw, va_raw, te_raw = split_data(sub)
IMPUTER = fit_imputer(tr_raw)
S = {"tr": apply_imputer(tr_raw, IMPUTER), "va": apply_imputer(va_raw, IMPUTER), "te": apply_imputer(te_raw, IMPUTER)}
print({k: v.shape for k, v in S.items()})
print("Nulos restantes:", sum(v.isna().sum().sum() for v in S.values()))

### 1.5 Outliers, balanceo, codificación y escalado
**Decisiones de diseño (documentar en el PDF):**
* **Detección:** IQR con factor 1.5 sobre las 12 variables numéricas, límites calculados en train.
* **Sin outliers:** los valores detectados se *recortan* (winsorizing) a los límites IQR en train (eliminar filas quitaría demasiados datos: la contaminación tiene colas pesadas).
* **Con outliers:** sobre el train ya recortado se inyecta sintéticamente un 5 % de filas con un valor fuera del límite superior IQR en una variable numérica aleatoria → exactamente 5 % de outliers.
* **Val y test** se recortan siempre con los límites aprendidos en train (en los 8 casos): el test queda idéntico en valores y registros, y lo único que cambia entre casos es cómo se entrena el modelo.
* **Desbalanceado:** submuestreo de train hasta que la clase mayoritaria sea ≈70 % y las demás queden iguales.
* **Balanceado:** SMOTE (o sobremuestreo aleatorio) sobre el train natural hasta igualar todas las clases (SMOTE se ejecuta en espacio estandarizado temporal y luego se revierte; las dummies se redondean a 0/1).
* **Categóricas:** One-Hot para `City` y `Season` (nominales, sin orden); ordinal para `Year` (orden temporal).
* **Escalado:** StandardScaler (por defecto) sobre numéricas + ordinal, ajustado solo con train; es menos distorsionado por la dispersión de las variables que MinMax cuando existen valores extremos.

In [ ]:
def iqr_bounds(df, cols, factor=1.5):
    q1, q3 = df[cols].quantile(0.25), df[cols].quantile(0.75)
    iqr = q3 - q1
    return pd.DataFrame({"lo": q1 - factor * iqr, "hi": q3 + factor * iqr, "iqr": iqr})


def outlier_rate(df, bounds):
    """Porcentaje de filas con al menos un outlier IQR."""
    flag = (df[NUM_COLS] < bounds["lo"]) | (df[NUM_COLS] > bounds["hi"])
    return 100 * flag.any(axis=1).mean(), flag.mean() * 100


BOUNDS = iqr_bounds(S["tr"], NUM_COLS)
tasa, por_var = outlier_rate(S["tr"], BOUNDS)
print(f"Train natural: {tasa:.1f}% de filas con algún outlier IQR (1.5). Por variable (%):")
print(por_var.round(1).to_string())


def clip_outliers(df, bounds):
    df = df.copy()
    df[NUM_COLS] = df[NUM_COLS].clip(lower=bounds["lo"], upper=bounds["hi"], axis=1)
    return df


def inject_outliers(X, bounds, frac=0.05, seed=SEED):
    """Inyecta outliers sintéticos en el `frac` de las filas (variable numérica aleatoria, por encima del límite superior)."""
    rng = np.random.default_rng(seed)
    X = X.copy()
    n = int(round(frac * len(X)))
    idx = rng.choice(len(X), n, replace=False)
    cols = rng.choice(NUM_COLS, n)
    for i, c in zip(idx, cols):
        X.iloc[i, X.columns.get_loc(c)] = bounds.loc[c, "hi"] + rng.uniform(1.5, 4.0) * max(bounds.loc[c, "iqr"], 1e-6)
    return X, {"n_inyectados": n, "pct": 100 * n / len(X), "por_variable": pd.Series(cols).value_counts().to_dict()}


def make_imbalanced(df, major_share=0.70, seed=SEED):
    """Submuestrea para que la clase mayoritaria sea ~major_share y las demás queden iguales."""
    counts = df[TARGET].value_counts()
    major, others = counts.idxmax(), counts.drop(counts.idxmax())
    k = int(others.min())
    n_major = int(round(k * len(others) * major_share / (1 - major_share)))
    if n_major > counts[major]:                       # no hay suficientes de la mayoritaria: se reduce el resto
        n_major = int(counts[major])
        k = int(n_major * (1 - major_share) / major_share / len(others))
    parts = [df[df[TARGET] == major].sample(n_major, random_state=seed)]
    parts += [df[df[TARGET] == c].sample(k, random_state=seed) for c in others.index]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


class Encoder:
    """One-Hot (nominales) + ordinal (Year) + numéricas; se ajusta solo con train."""
    def fit(self, df):
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(df[NOMINAL_COLS])
        self.ohe_cols = list(self.ohe.get_feature_names_out(NOMINAL_COLS))
        return self

    def transform(self, df):
        out = df[NUM_COLS].reset_index(drop=True).copy()
        for c in ORDINAL_COLS:
            out[c] = df[c].map({v: i for i, v in enumerate(YEARS_ORDER)}).fillna(-1).values
        dummies = pd.DataFrame(self.ohe.transform(df[NOMINAL_COLS]), columns=self.ohe_cols)
        return pd.concat([out, dummies], axis=1)


def balance_train(X, y, ohe_cols, method=BALANCE_METHOD, seed=SEED):
    if method == "ros":
        Xr, yr = RandomOverSampler(random_state=seed).fit_resample(X, y)
        return Xr.reset_index(drop=True), yr
    sc = StandardScaler().fit(X)
    Zr, yr = SMOTE(random_state=seed, k_neighbors=5).fit_resample(sc.transform(X), y)
    Xr = pd.DataFrame(sc.inverse_transform(Zr), columns=X.columns)
    Xr[ohe_cols] = (Xr[ohe_cols] > 0.5).astype(float)
    return Xr, yr


def prepare_case(cfg, S=S, seed=SEED):
    """Construye X/y de train, val y test para una combinación (escalado, outliers, balanceo)."""
    tr = S["tr"].copy()
    info = {}
    if not cfg["balance"]:
        tr = make_imbalanced(tr, seed=seed)
    tr = clip_outliers(tr, BOUNDS)
    enc = Encoder().fit(tr)
    Xtr, ytr = enc.transform(tr), tr[TARGET].map(CLASS_TO_ID).values
    info["dist_antes_balanceo"] = pd.Series(ytr).map(dict(enumerate(CLASES))).value_counts().to_dict()
    if cfg["balance"]:
        Xtr, ytr = balance_train(Xtr, ytr, enc.ohe_cols, seed=seed)
    if cfg["outliers"]:
        Xtr, info["outliers"] = inject_outliers(Xtr, BOUNDS, 0.05, seed)
    # Val y test reciben SIEMPRE el mismo tratamiento (recorte con los límites IQR aprendidos en train), sea cual sea el caso:
    # así el test es idéntico en los 8 casos y solo cambia cómo se entrena. (Sin esto, el test conservaba valores 10-100 veces
    # más grandes que cualquier cosa vista en train y los modelos sensibles a la escala fallaban por ese desajuste.)
    Xva, Xte = enc.transform(clip_outliers(S["va"], BOUNDS)), enc.transform(clip_outliers(S["te"], BOUNDS))
    yva, yte = S["va"][TARGET].map(CLASS_TO_ID).values, S["te"][TARGET].map(CLASS_TO_ID).values
    if cfg["scale"]:
        sc_cols = NUM_COLS + ORDINAL_COLS
        scaler = (StandardScaler() if SCALER == "standard" else MinMaxScaler()).fit(Xtr[sc_cols])
        for X in (Xtr, Xva, Xte):
            X[sc_cols] = scaler.transform(X[sc_cols])
    info["dist_train"] = pd.Series(ytr).map(dict(enumerate(CLASES))).value_counts().to_dict()
    info["n_train"] = len(Xtr)
    return dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte, cols=list(Xtr.columns), info=info,
                te_index=S["te"].index.values)


CASES = {i: prepare_case(cfg) for i, cfg in CASOS.items()}
for i, d in CASES.items():
    print(f"Caso {i}: train={d['info']['n_train']} features={d['Xtr'].shape[1]} dist={d['info']['dist_train']}"
          + (f" outliers={d['info']['outliers']['n_inyectados']} ({d['info']['outliers']['pct']:.1f}%)" if "outliers" in d["info"] else ""))
# Verificación: el conjunto de prueba es el mismo en los 8 casos
assert all((CASES[i]["te_index"] == CASES[1]["te_index"]).all() and (CASES[i]["yte"] == CASES[1]["yte"]).all() for i in CASES)
print("OK: test idéntico en los 8 casos")

## 2. Aprendizaje supervisado
### 2.1 Funciones de entrenamiento y evaluación
Métricas: accuracy y precision/recall/F1 **macro** (promedio sin ponderar por clase → penaliza ignorar clases minoritarias).
Hiperparámetros: árbol (criterio y `max_depth` elegidos con validación por F1), KNN (k por CV de 5 folds), SVM (RBF, C=1, gamma='scale'),
red neuronal (MLP con regularización L2 y Dropout, y callback EarlyStopping, tal como en el taller 8 de clase).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
try:
    tf.config.experimental.enable_op_determinism()   # resultados reproducibles con la misma semilla
except Exception as e:
    print("No se pudo activar el determinismo de TF:", e)


def calc_metrics(y, pred):
    return dict(accuracy=accuracy_score(y, pred),
                precision=precision_score(y, pred, average="macro", zero_division=0),
                recall=recall_score(y, pred, average="macro", zero_division=0),
                f1=f1_score(y, pred, average="macro", zero_division=0))


def train_tree(d, seed=SEED):
    depths = [2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
    curves, best, best_gini = {}, None, None
    for crit in ["gini", "entropy"]:
        tr_acc, va_acc = [], []
        for md in depths:
            m = DecisionTreeClassifier(criterion=crit, max_depth=md, random_state=seed).fit(d["Xtr"], d["ytr"])
            tr_acc.append(m.score(d["Xtr"], d["ytr"])); va_acc.append(m.score(d["Xva"], d["yva"]))
            f1v = f1_score(d["yva"], m.predict(d["Xva"]), average="macro", zero_division=0)
            if best is None or f1v > best[0]: best = (f1v, m, crit, md)
            if crit == "gini" and (best_gini is None or f1v > best_gini[0]): best_gini = (f1v, m, md)
        curves[crit] = dict(depths=depths, train=tr_acc, val=va_acc)
    return dict(model=best[1], criterion=best[2], depth=best[3], curves=curves, gini_model=best_gini[1])


def train_knn(d, seed=SEED):
    ks = list(range(1, 26, 2))
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    accs = [cross_val_score(KNeighborsClassifier(n_neighbors=k), d["Xtr"], d["ytr"], cv=cv).mean() for k in ks]
    k_best = ks[int(np.argmax(accs))]
    return dict(model=KNeighborsClassifier(n_neighbors=k_best).fit(d["Xtr"], d["ytr"]), ks=ks, cv_acc=accs, k=k_best)


def train_svm(d, seed=SEED):
    return dict(model=SVC(kernel="rbf", C=1.0, gamma="scale", decision_function_shape="ovr", random_state=seed).fit(d["Xtr"], d["ytr"]))


def build_nn(n_in, n_out, seed=SEED):
    """MLP como en clase (taller 8): Dense + ReLU con regularización L2, Dropout y salida softmax; Adam + entropía cruzada categórica."""
    tf.keras.utils.set_random_seed(seed)
    reg = l2(1e-3)
    m = Sequential([
        Input(shape=(n_in,)),
        Dense(64, activation="relu", kernel_regularizer=reg), Dropout(0.3),
        Dense(32, activation="relu", kernel_regularizer=reg), Dropout(0.3),
        Dense(n_out, activation="softmax")])
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m


def train_nn(d, seed=SEED):
    """Entrena con etiquetas one-hot, validación explícita y EarlyStopping sobre val_loss (patience=10, restaura los mejores pesos)."""
    m = build_nn(d["Xtr"].shape[1], K_CLASES, seed)
    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    h = m.fit(d["Xtr"].values.astype("float32"), to_categorical(d["ytr"], K_CLASES),
              validation_data=(d["Xva"].values.astype("float32"), to_categorical(d["yva"], K_CLASES)),
              epochs=200, batch_size=64, callbacks=[early_stopping], verbose=0)
    return dict(model=m, history=h.history)


TRAINERS = {"Árboles": train_tree, "KNN": train_knn, "SVM": train_svm, "Redes Neuronales": train_nn}


def predict_scores(name, model, X):
    """Devuelve (predicción, puntajes por clase) para curvas ROC/PR."""
    if name == "Redes Neuronales":
        p = model.predict(X.values.astype("float32"), verbose=0)
        return p.argmax(1), p
    if name == "SVM":
        s = model.decision_function(X)
        return s.argmax(1), s
    return model.predict(X), model.predict_proba(X)

### 2.1.1 Arquitectura y configuración de la red neuronal (MLP)
* **Entrada:** todas las variables preprocesadas de cada caso (12 numéricas + `Year` + dummies de `City` y `Season`, unas 43 columnas).
* **Capas:** Dense(64, ReLU, L2=1e-3) → Dropout(0.3) → Dense(32, ReLU, L2=1e-3) → Dropout(0.3) → Dense(6, softmax).
* **Regularización (vista en clase):** L2 en las capas ocultas, Dropout (0.3) y Early Stopping.
* **Entrenamiento:** Adam, pérdida `categorical_crossentropy` con etiquetas one-hot (`to_categorical`), lotes de 64, hasta 200 épocas.
* **Callback (visto en clase):** `EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)`.
* **Validación:** el conjunto de validación (15 %) se pasa en `validation_data`; el test no se toca hasta evaluar.
* **Reproducibilidad:** `tf.keras.utils.set_random_seed(SEED)` y operaciones deterministas de TensorFlow.
* **Proceso de configuración:** arrancamos con una red pequeña y le fuimos subiendo regularización hasta que las curvas de train y validación dejaron de separarse (ver curvas del caso 8).

### 2.2 Entrenamiento de los 32 experimentos (8 casos × 4 modelos)

In [ ]:
FITTED, OUTPUTS, rows = {}, {}, []
for cid, d in CASES.items():
    for name, trainer in TRAINERS.items():
        set_seeds()
        fit = trainer(d)
        pred, score = predict_scores(name, fit["model"], d["Xte"])
        FITTED[(cid, name)] = fit
        OUTPUTS[(cid, name)] = (pred, score)
        rows.append(dict(caso=cid, modelo=name, **calc_metrics(d["yte"], pred)))
    print("Caso", cid, "listo")
RES = pd.DataFrame(rows)
RES.to_csv(os.path.join(OUT_DIR, "resultados_supervisado.csv"), index=False)

### 2.2.1 Evaluación sobre 3 datos de prueba
Elegimos 3 registros del conjunto de prueba (los mismos en los 8 casos) y comparamos la clase real con lo que predice cada técnica.

In [ ]:
IDX3 = np.random.default_rng(SEED).choice(len(CASES[1]["yte"]), 3, replace=False)
rows3 = []
for cid, d in CASES.items():
    for j in IDX3:
        r = {"caso": cid, "dato_prueba": int(j), "real": CLASES[d["yte"][j]]}
        for name in TRAINERS:
            r[name] = CLASES[OUTPUTS[(cid, name)][0][j]]
        rows3.append(r)
TRES_DATOS = pd.DataFrame(rows3)
TRES_DATOS.to_csv(os.path.join(OUT_DIR, "tres_datos_prueba_supervisado.csv"), index=False)
TRES_DATOS

### 2.3 Evaluación: tabla de la figura 1

In [ ]:
def fig1_table(res):
    t = res.pivot(index="caso", columns="modelo", values=["accuracy", "precision", "recall", "f1"])
    t = t.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product(
        [list(TRAINERS), ["accuracy", "precision", "recall", "f1"]])).round(4)
    cfg = pd.DataFrame({"ED": ["SI" if v["scale"] else "NO" for v in CASOS.values()],
                        "Outliers": ["SI" if v["outliers"] else "NO" for v in CASOS.values()],
                        "Balanceo": ["SI" if v["balance"] else "NO" for v in CASOS.values()]}, index=list(CASOS))
    cfg.columns = pd.MultiIndex.from_product([["Preproc."], cfg.columns])
    return pd.concat([cfg, t], axis=1)


FIG1 = fig1_table(RES)
FIG1.to_csv(os.path.join(OUT_DIR, "figura1_tabla.csv"))
FIG1

In [ ]:
# Gráficas pedidas: F1 máximo por algoritmo y F1 medio por escalado / outliers / balanceo
R = RES.assign(ED=lambda t: t.caso.map(lambda c: "SI" if CASOS[c]["scale"] else "NO"),
               Outliers=lambda t: t.caso.map(lambda c: "SI" if CASOS[c]["outliers"] else "NO"),
               Balanceo=lambda t: t.caso.map(lambda c: "SI" if CASOS[c]["balance"] else "NO"))
order = list(TRAINERS)

fig, ax = plt.subplots(figsize=(6.5, 4))
mx = R.groupby("modelo").f1.max().reindex(order)
ax.bar(order, mx.values, color=sns.color_palette("deep", 4))
for i, v in enumerate(mx.values): ax.text(i, v + 0.005, f"{v:.3f}", ha="center")
ax.set(ylabel="F1 macro máximo", title="F1-score máximo por algoritmo")
savefig("f1_max_algoritmo.png")

for var, title in [("ED", "escalado de datos (ED)"), ("Outliers", "manejo de outliers"), ("Balanceo", "balanceo de clases")]:
    t = R.groupby(["modelo", var]).f1.mean().unstack().reindex(order)
    t.plot(kind="bar", figsize=(7, 4), rot=0, color=["#4c72b0", "#dd8452"])
    plt.ylabel("F1 macro medio"); plt.title(f"F1-score medio por {title} y algoritmo"); plt.legend(title=var)
    savefig(f"f1_medio_{var.lower()}.png")

### 2.3.1 Matrices de confusión y reporte por clase (caso 8)

In [ ]:
d8 = CASES[8]
fig, axs = plt.subplots(2, 2, figsize=(11, 9))
for ax, name in zip(axs.ravel(), TRAINERS):
    pred = OUTPUTS[(8, name)][0]
    ConfusionMatrixDisplay(confusion_matrix(d8["yte"], pred, labels=range(K_CLASES)), display_labels=CLASES).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"{name} – caso 8")
savefig("matrices_confusion_caso8.png")
for name in TRAINERS:
    print(f"\n=== {name} (caso 8) ===")
    print(classification_report(d8["yte"], OUTPUTS[(8, name)][0], target_names=CLASES, zero_division=0))

### 2.3.2 Curvas ROC y Precision-Recall (caso 8, One-vs-Rest con promedio micro; AUC macro OvR y AUC-PR en la leyenda)

In [ ]:
Y8 = label_binarize(d8["yte"], classes=range(K_CLASES))
ROC_STATS = {}
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for name in TRAINERS:
    s = OUTPUTS[(8, name)][1]
    fpr, tpr, _ = roc_curve(Y8.ravel(), s.ravel())
    p, r, _ = precision_recall_curve(Y8.ravel(), s.ravel())
    ROC_STATS[name] = dict(auc_macro=float(roc_auc_score(Y8, s, average="macro")), auc_pr_micro=float(auc(r, p)))
    axs[0].plot(fpr, tpr, label=f"{name} (AUC macro={ROC_STATS[name]['auc_macro']:.3f})")
    axs[1].plot(r, p, label=f"{name} (AUC-PR micro={ROC_STATS[name]['auc_pr_micro']:.3f})")
axs[0].plot([0, 1], [0, 1], "k--", lw=0.8); axs[0].set(xlabel="FPR", ylabel="TPR", title="ROC (micro) – caso 8")
axs[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (micro) – caso 8")
axs[0].legend(); axs[1].legend()
savefig("roc_pr_caso8.png")

### 2.4 Visualización de resultados
PCA (2 componentes) se usa **solo para graficar**; los modelos se entrenan con todas las variables.

In [ ]:
def viz_arbol(cid, show=True):
    d, f = CASES[cid], FITTED[(cid, "Árboles")]
    imp = pd.Series(f["model"].feature_importances_, index=d["cols"]).sort_values(ascending=False).head(12)
    fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
    imp[::-1].plot(kind="barh", ax=axs[0], color="seagreen", title=f"Importancia de características (caso {cid})")
    for crit, ls in [("gini", "-"), ("entropy", "--")]:
        c = f["curves"][crit]
        axs[1].plot(c["depths"], c["train"], ls, color="C0", label=f"train {crit}")
        axs[1].plot(c["depths"], c["val"], ls, color="C1", label=f"val {crit}")
    axs[1].set(xlabel="max_depth", ylabel="accuracy", title="Subajuste / sobreajuste: Gini vs Entropía"); axs[1].legend()
    plot_tree(f["gini_model"], max_depth=2, feature_names=d["cols"], class_names=CLASES, filled=True, fontsize=6, ax=axs[2])
    axs[2].set_title("Árbol (criterio Gini, 2 primeros niveles)")
    savefig(f"arbol_caso{cid}.png", show)
    reglas = export_text(f["gini_model"], feature_names=d["cols"], max_depth=3)
    open(os.path.join(OUT_DIR, f"reglas_arbol_caso{cid}.txt"), "w", encoding="utf-8").write(reglas)
    if show:
        print(f"Caso {cid}: mejor criterio={f['criterion']}, max_depth={f['depth']}")
        print(reglas)


def viz_knn(cid, n_test=3, seed=SEED, show=True):
    d, f = CASES[cid], FITTED[(cid, "KNN")]
    fig, axs = plt.subplots(1, 2, figsize=(15, 5.5))
    axs[0].plot(f["ks"], f["cv_acc"], "o-"); axs[0].axvline(f["k"], color="r", ls="--", label=f"k elegido = {f['k']}")
    axs[0].set(xlabel="k", ylabel="accuracy (CV 5 folds)", title=f"k vs accuracy – caso {cid}"); axs[0].legend()
    pca = PCA(2, random_state=seed).fit(d["Xtr"])
    Ptr, Pte = pca.transform(d["Xtr"]), pca.transform(d["Xte"])
    axs[1].scatter(Ptr[:, 0], Ptr[:, 1], c=d["ytr"], cmap="tab10", s=6, alpha=0.35)
    # Límites robustos: los datos de prueba conservan sus outliers y pueden caer muy lejos de la nube de entrenamiento
    lo, hi = np.percentile(Ptr, 0.5, axis=0), np.percentile(Ptr, 99.5, axis=0)
    margen = 0.05 * (hi - lo)
    ok = np.where(((Pte >= lo) & (Pte <= hi)).all(axis=1))[0]
    rng = np.random.default_rng(seed)
    for i in rng.choice(ok, min(n_test, len(ok)), replace=False):
        _, idx = f["model"].kneighbors(d["Xte"].iloc[[i]])   # vecinos calculados en el espacio completo
        axs[1].scatter(*Pte[i], marker="*", s=250, c="k")
        for j in idx[0]:
            axs[1].plot([Pte[i, 0], Ptr[j, 0]], [Pte[i, 1], Ptr[j, 1]], "k-", lw=0.6)
            axs[1].scatter(*Ptr[j], s=30, edgecolor="k", facecolor="none")
    axs[1].set_xlim(lo[0] - margen[0], hi[0] + margen[0]); axs[1].set_ylim(lo[1] - margen[1], hi[1] + margen[1])
    axs[1].set_title("Vecinos (PCA 2D) de 3 puntos de prueba (★)")
    savefig(f"knn_caso{cid}.png", show)


def viz_svm(cid, seed=SEED, show=True):
    """Regiones de decisión en PCA 2D: se reentrena un SVM (misma config.) SOLO sobre la proyección, a modo de visualización."""
    d = CASES[cid]
    pca = PCA(2, random_state=seed).fit(d["Xtr"])
    Ptr, Pte = pca.transform(d["Xtr"]), pca.transform(d["Xte"])
    m = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed).fit(Ptr, d["ytr"])
    lo, hi = np.percentile(Ptr, 0.5, axis=0), np.percentile(Ptr, 99.5, axis=0)   # límites robustos (ver viz_knn)
    mg = 0.05 * (hi - lo); lo, hi = lo - mg, hi + mg
    xx, yy = np.meshgrid(np.linspace(lo[0], hi[0], 250), np.linspace(lo[1], hi[1], 250))
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap="tab10", levels=np.arange(-0.5, K_CLASES + 0.5))
    vis = ((Pte >= lo) & (Pte <= hi)).all(axis=1)
    plt.scatter(Pte[vis, 0], Pte[vis, 1], c=d["yte"][vis], cmap="tab10", s=8, edgecolor="k", linewidth=0.2)
    plt.title(f"SVM – regiones de decisión (PCA 2D) – caso {cid}"); plt.xlabel("PC1"); plt.ylabel("PC2")
    savefig(f"svm_regiones_caso{cid}.png", show)


for cid in (CASES if VIZ_ALL_CASES else [8]):
    viz_arbol(cid, show=(cid == 8)); viz_knn(cid, show=(cid == 8)); viz_svm(cid, show=(cid == 8))

In [ ]:
# Curvas de entrenamiento de la red neuronal (caso 8) – útil para documentar regularización y callbacks
h = FITTED[(8, "Redes Neuronales")]["history"]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].plot(h["loss"], label="train"); axs[0].plot(h["val_loss"], label="val"); axs[0].set(title="Pérdida (EarlyStopping)", xlabel="época"); axs[0].legend()
axs[1].plot(h["accuracy"], label="train"); axs[1].plot(h["val_accuracy"], label="val"); axs[1].set(title="Accuracy", xlabel="época"); axs[1].legend()
savefig("nn_curvas_caso8.png")
FITTED[(8, "Redes Neuronales")]["model"].summary()

## 3. Aprendizaje no supervisado
Se usa el mismo dataset **sin la etiqueta**. Los 8 casos de la figura 2 se construyen con las mismas versiones de train de la parte
supervisada (mismos registros, mismo escalado/outliers/balanceo) y luego se descarta la variable objetivo.
La etiqueta original solo se usa para **validación externa** (cruce cluster vs clase real), nunca para entrenar.

### 3.1 Utilidades: punto de codo, K-Means y DBSCAN

In [ ]:
def knee_index(y):
    """Punto de máxima distancia a la cuerda que une el primer y último punto de la curva (método del codo)."""
    y = np.asarray(y, float); x = np.arange(len(y))
    xn = (x - x[0]) / (x[-1] - x[0]); yn = (y - y.min()) / (y.max() - y.min() + 1e-12)
    return int(np.argmax(np.abs(yn - (yn[0] + (yn[-1] - yn[0]) * xn))))


def sil(X, labels, seed=SEED, n=3000):
    return float(silhouette_score(X, labels, sample_size=min(n, len(X)), random_state=seed))


def run_kmeans(X, ks=range(2, 11), seed=SEED):
    inertia, sils, models = [], [], {}
    for k in ks:
        km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=seed).fit(X)
        inertia.append(km.inertia_); sils.append(sil(X, km.labels_, seed)); models[k] = km
    ks = list(ks); k_opt = ks[int(np.argmax(sils))]
    return dict(ks=ks, inertia=inertia, sil=sils, k_opt=k_opt, k_elbow=ks[knee_index(inertia)], model=models[k_opt],
                sizes=np.bincount(models[k_opt].labels_).tolist())


def run_dbscan(X, seed=SEED):
    min_pts = 2 * X.shape[1]                                        # regla general: 2 × nº de características
    kd = np.sort(NearestNeighbors(n_neighbors=min_pts).fit(X).kneighbors(X)[0][:, -1])
    eps_knee = float(kd[knee_index(kd)])

    def fit_eps(eps):
        db = DBSCAN(eps=eps, min_samples=min_pts).fit(X)
        mask = db.labels_ != -1
        n_cl = len(set(db.labels_[mask]))
        s = sil(X[mask], db.labels_[mask], seed) if n_cl >= 2 and mask.sum() > n_cl else np.nan
        return dict(eps=eps, model=db, n_clusters=n_cl, noise_pct=100 * (~mask).mean(), sil=s)

    best = fit_eps(eps_knee)
    # Si con ε del codo se fusiona todo en 1 cluster (silhouette no calculable), se explora ε <= ε_codo
    # y se elige el de mayor silhouette con >=2 clusters y <=50 % de ruido (se reporta ambos ε).
    if best["n_clusters"] < 2:
        cands = [r for r in (fit_eps(e) for e in np.linspace(0.3 * eps_knee, 0.95 * eps_knee, 12))
                 if r["n_clusters"] >= 2 and r["noise_pct"] <= 50 and not np.isnan(r["sil"])]
        if cands:
            best = max(cands, key=lambda r: r["sil"])
    return dict(kdist=kd, eps_knee=eps_knee, min_pts=min_pts, **best)

### 3.2 Entrenamiento y 3.3 Evaluación (tabla de la figura 3)

In [ ]:
UNSUP, rows = {}, []
for cid, d in CASES.items():
    X = d["Xtr"].values                      # sin etiqueta
    y_ext = d["ytr"]                         # SOLO validación externa
    km, db = run_kmeans(X), run_dbscan(X)
    UNSUP[cid] = dict(km=km, db=db)
    rows.append({"#": cid, "ED": "SI" if CASOS[cid]["scale"] else "NO", "Outliers": "SI" if CASOS[cid]["outliers"] else "NO",
                 "Balanceo": "SI" if CASOS[cid]["balance"] else "NO",
                 "K óptimo (silhouette)": km["k_opt"], "K codo": km["k_elbow"],
                 "KM inertia": round(km["inertia"][km["ks"].index(km["k_opt"])], 1), "KM silhouette": round(max(km["sil"]), 4),
                 "KM tamaños": km["sizes"],
                 "DB eps codo": round(db["eps_knee"], 3), "DB eps usado": round(db["eps"], 3), "DB MinPts": db["min_pts"], "DB clusters": db["n_clusters"],
                 "DB ruido %": round(db["noise_pct"], 1), "DB silhouette": None if np.isnan(db["sil"]) else round(db["sil"], 4)})
    # Validación externa (solo para interpretar): cruce de cada cluster con la clase real
    for algo, lab in [("kmeans", km["model"].labels_), ("dbscan", db["model"].labels_)]:
        pd.crosstab(pd.Series(y_ext).map(dict(enumerate(CLASES))).values, lab).reindex(CLASES).to_csv(
            os.path.join(OUT_DIR, f"crosstab_caso{cid}_{algo}.csv"))
    print("Caso", cid, "listo")
FIG3 = pd.DataFrame(rows).set_index("#")
FIG3.to_csv(os.path.join(OUT_DIR, "figura3_tabla.csv"))
FIG3

#### 3.2.1 Asignación de 3 datos de prueba a los clusters
K-Means asigna al centroide más cercano. DBSCAN no tiene `predict`, así que cada dato toma el cluster del punto núcleo más cercano
si queda a distancia ≤ ε; si no, se marca como ruido (-1). La clase real aparece solo como referencia.

In [ ]:
def dbscan_predict(db, Xtrain, Xnew):
    """Asigna puntos nuevos a un cluster de DBSCAN (punto núcleo más cercano a distancia <= eps) o ruido (-1)."""
    core, lab = Xtrain[db.core_sample_indices_], db.labels_[db.core_sample_indices_]
    if len(core) == 0:
        return -np.ones(len(Xnew), int)
    dist, idx = NearestNeighbors(n_neighbors=1).fit(core).kneighbors(Xnew)
    out = lab[idx[:, 0]].copy(); out[dist[:, 0] > db.eps] = -1
    return out


rows3u = []
for cid, d in CASES.items():
    Xn = d["Xte"].values[IDX3]
    kl = UNSUP[cid]["km"]["model"].predict(Xn)
    dl = dbscan_predict(UNSUP[cid]["db"]["model"], d["Xtr"].values, Xn)
    for j, a, b in zip(IDX3, kl, dl):
        rows3u.append({"caso": cid, "dato_prueba": int(j), "clase real (solo referencia)": CLASES[d["yte"][j]],
                       "cluster K-Means": int(a), "cluster DBSCAN (-1 = ruido)": int(b)})
TRES_DATOS_U = pd.DataFrame(rows3u)
TRES_DATOS_U.to_csv(os.path.join(OUT_DIR, "tres_datos_prueba_no_supervisado.csv"), index=False)
TRES_DATOS_U

In [ ]:
# Codo (inertia vs K), silhouette vs K y k-distancia para DBSCAN, por caso
fig, axs = plt.subplots(8, 3, figsize=(15, 26))
for cid in CASES:
    u = UNSUP[cid]
    axs[cid - 1, 0].plot(u["km"]["ks"], u["km"]["inertia"], "o-"); axs[cid - 1, 0].axvline(u["km"]["k_elbow"], color="r", ls="--")
    axs[cid - 1, 0].set(title=f"Caso {cid} – codo (inertia)", xlabel="K")
    axs[cid - 1, 1].plot(u["km"]["ks"], u["km"]["sil"], "o-", color="C1"); axs[cid - 1, 1].axvline(u["km"]["k_opt"], color="r", ls="--")
    axs[cid - 1, 1].set(title=f"Caso {cid} – silhouette (K óptimo={u['km']['k_opt']})", xlabel="K")
    axs[cid - 1, 2].plot(u["db"]["kdist"]); axs[cid - 1, 2].axhline(u["db"]["eps_knee"], color="r", ls="--"); axs[cid - 1, 2].axhline(u["db"]["eps"], color="g", ls=":")
    axs[cid - 1, 2].set(title=f"Caso {cid} – k-distancia (k={u['db']['min_pts']}, ε codo={u['db']['eps_knee']:.2f}, usado={u['db']['eps']:.2f})")
savefig("kmeans_dbscan_seleccion_parametros.png")

In [ ]:
# Barras de silhouette por algoritmo (8 casos cada una)
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].bar(FIG3.index.astype(str), FIG3["KM silhouette"], color="C0"); axs[0].set(title="K-Means – silhouette por caso", xlabel="caso")
axs[1].bar(FIG3.index.astype(str), FIG3["DB silhouette"].astype(float).fillna(0), color="C2"); axs[1].set(title="DBSCAN – silhouette por caso (0 = no calculable)", xlabel="caso")
savefig("silhouette_por_caso.png")

### 3.4 Visualización de clusters en 2D (PCA); centroides en K-Means y ruido en DBSCAN con color distinto

In [ ]:
for cid, d in CASES.items():
    X = d["Xtr"].values
    pca = PCA(2, random_state=SEED).fit(X); P = pca.transform(X)
    km, db = UNSUP[cid]["km"], UNSUP[cid]["db"]
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    axs[0].scatter(P[:, 0], P[:, 1], c=km["model"].labels_, cmap="tab10", s=6, alpha=0.6)
    C = pca.transform(km["model"].cluster_centers_)
    axs[0].scatter(C[:, 0], C[:, 1], c="red", marker="X", s=220, edgecolor="k", label="centroides"); axs[0].legend()
    axs[0].set_title(f"Caso {cid} – K-Means (K={km['k_opt']})")
    lab = db["model"].labels_
    axs[1].scatter(P[lab == -1, 0], P[lab == -1, 1], c="lightgray", s=6, label="ruido")
    axs[1].scatter(P[lab != -1, 0], P[lab != -1, 1], c=lab[lab != -1], cmap="tab10", s=6)
    axs[1].set_title(f"Caso {cid} – DBSCAN (ε={db['eps']:.2f}, {db['n_clusters']} clusters, ruido {db['noise_pct']:.0f}%)"); axs[1].legend()
    savefig(f"clusters_caso{cid}.png")

## 4. Entregables
* Guardar `salidas/` (tablas CSV y figuras) para insertarlas en el documento de diseño (PDF).
* Verificar que el notebook corre de principio a fin y que `CEDULA` esté declarada.
* Nombre del ZIP: `practica2-grupo-XX-equipo-YY.zip`.

In [ ]:
# Exporta todo lo necesario para el informe (cifras, hiperparámetros elegidos, reportes por clase) y empaqueta salidas/
import json
d8 = CASES[8]
for name in TRAINERS:
    rep_ = classification_report(d8["yte"], OUTPUTS[(8, name)][0], target_names=CLASES, zero_division=0, output_dict=True)
    pd.DataFrame(rep_).T.round(4).to_csv(os.path.join(OUT_DIR, f"reporte_clases_caso8_{name.replace(' ', '_')}.csv"))
    pd.DataFrame(confusion_matrix(d8["yte"], OUTPUTS[(8, name)][0], labels=range(K_CLASES)), index=CLASES, columns=CLASES).to_csv(
        os.path.join(OUT_DIR, f"matriz_confusion_caso8_{name.replace(' ', '_')}.csv"))

corr = sub[NUM_COLS].corr().abs().where(np.triu(np.ones((len(NUM_COLS),) * 2), 1).astype(bool)).stack()
RESUMEN = dict(
    cedula=CEDULA, grupo=GRUPO, equipo=EQUIPO, n_registros=N_REGISTROS, seed=SEED, scaler=SCALER, balance_method=BALANCE_METHOD,
    filas_originales=int(len(raw)), filas_etiquetadas=int(len(labeled)),
    dist_subconjunto=sub[TARGET].value_counts().reindex(CLASES).to_dict(),
    nulos_pct={c: round(float(sub[c].isna().mean() * 100), 2) for c in NUM_COLS},
    outliers_train_natural_filas_pct=round(float(tasa), 1), outliers_por_variable_pct=por_var.round(1).to_dict(),
    correlaciones_altas={f"{a} - {b}": round(float(v), 2) for (a, b), v in corr[corr > 0.8].items()},
    tamanos_split={k: int(len(v)) for k, v in S.items()},
    casos={cid: dict(n_train=d["info"]["n_train"], n_features=int(d["Xtr"].shape[1]), dist_train=d["info"]["dist_train"],
                     outliers=d["info"].get("outliers")) for cid, d in CASES.items()},
    arbol={cid: dict(criterio=FITTED[(cid, "Árboles")]["criterion"], max_depth=FITTED[(cid, "Árboles")]["depth"],
                     top_features=pd.Series(FITTED[(cid, "Árboles")]["model"].feature_importances_, index=CASES[cid]["cols"]).nlargest(5).round(3).to_dict())
           for cid in CASES},
    knn_k={cid: FITTED[(cid, "KNN")]["k"] for cid in CASES},
    nn_epocas={cid: len(FITTED[(cid, "Redes Neuronales")]["history"]["loss"]) for cid in CASES},
    roc_caso8=ROC_STATS,
)
json.dump(RESUMEN, open(os.path.join(OUT_DIR, "resumen.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=str)

# Empaquetado de salidas y descarga automática si estamos en Colab
with zipfile.ZipFile("salidas.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
        z.write(f)
print(f"Cédula usada: {CEDULA} | N={N_REGISTROS} | random_state={SEED}")
print("Archivos generados:", len(os.listdir(OUT_DIR)), "en", OUT_DIR, "(también salidas.zip)")
try:
    from google.colab import files
    files.download("salidas.zip")
except ImportError:
    pass